In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction import FeatureHasher
from sklearn.metrics import accuracy_score
import hashlib

import warnings


# Create sample dataset with very high cardinality
np.random.seed(42)
n_samples = 1000

In [2]:
# Simulate very high cardinality features
user_ids = [f'user_{i}' for i in range(5000)]  # 5000 unique users
urls = [f'https://example.com/page_{i}' for i in range(2000)]  # 2000 unique URLs

data = pd.DataFrame({
    'user_id': np.random.choice(user_ids, n_samples),
    'url': np.random.choice(urls, n_samples),
    'clicks': np.random.randint(0, 100, n_samples),
    'converted': np.random.randint(0, 2, n_samples)
})

print("Feature Hashing Example")
print("="*60)
print(f"\nDataset shape: {data.shape}")
print(f"Unique users: {data['user_id'].nunique()}")
print(f"Unique URLs: {data['url'].nunique()}")
print(f"Total unique categories: {data['user_id'].nunique() + data['url'].nunique()}")

Feature Hashing Example

Dataset shape: (1000, 4)
Unique users: 907
Unique URLs: 780
Total unique categories: 1687


### Method 1: Manual Hashing Implementation

In [3]:
def hash_feature(value, n_features, use_sign=True):
    """
    Hash a categorical value to a bin index.
    
    Parameters:
    -----------
    value : str
        Category value to hash
    n_features : int
        Number of hash bins
    use_sign : bool
        If True, use signed hashing to reduce collisions
    
    Returns:
    --------
    tuple : (bin_index, sign)
    """
    # Use a hash function
    hash_val = int(hashlib.md5(str(value).encode()).hexdigest(), 16)
    
    # Get bin index
    bin_idx = hash_val % n_features
    
    # Get sign (reduces collision impact)
    sign = 1 if use_sign and (hash_val // n_features) % 2 == 0 else -1 if use_sign else 1
    
    return bin_idx, sign

In [4]:
def hash_encode_series(series, n_features=32, use_sign=True):
    """
    Apply feature hashing to a categorical series.
    
    Parameters:
    -----------
    series : pd.Series
        Categorical series to encode
    n_features : int
        Number of hash bins
    use_sign : bool
        Use signed hashing
    
    Returns:
    --------
    pd.DataFrame : Hashed features
    """
    # Initialize result
    result = pd.DataFrame(0, index=series.index, 
                         columns=[f'hash_{i}' for i in range(n_features)])
    
    # Hash each value
    for idx, value in series.items():
        bin_idx, sign = hash_feature(value, n_features, use_sign)
        result.loc[idx, f'hash_{bin_idx}'] = sign
    
    return result

In [5]:
print("\n" + "="*60)
print("Method 1: Manual Feature Hashing")
print("="*60)

# Apply hashing with different bin sizes
n_features_options = [16, 32, 64]

print("\nComparing different hash sizes:")
for n_feat in n_features_options:
    hashed = hash_encode_series(data['user_id'], n_features=n_feat)
    
    # Count collisions (simplified: bins with multiple unique users)
    user_to_bins = {}
    for idx, user in data['user_id'].items():
        bin_idx, _ = hash_feature(user, n_feat)
        if bin_idx not in user_to_bins:
            user_to_bins[bin_idx] = set()
        user_to_bins[bin_idx].add(user)
    
    collided_bins = sum(1 for users in user_to_bins.values() if len(users) > 1)
    
    print(f"\n  n_features = {n_feat}:")
    print(f"    Unique users: {data['user_id'].nunique()}")
    print(f"    Hash bins: {n_feat}")
    print(f"    Bins with collisions: {collided_bins}/{n_feat} ({collided_bins/n_feat*100:.1f}%)")
    print(f"    Ratio (users/bins): {data['user_id'].nunique()/n_feat:.2f}")


Method 1: Manual Feature Hashing

Comparing different hash sizes:

  n_features = 16:
    Unique users: 907
    Hash bins: 16
    Bins with collisions: 16/16 (100.0%)
    Ratio (users/bins): 56.69

  n_features = 32:
    Unique users: 907
    Hash bins: 32
    Bins with collisions: 32/32 (100.0%)
    Ratio (users/bins): 28.34

  n_features = 64:
    Unique users: 907
    Hash bins: 64
    Bins with collisions: 64/64 (100.0%)
    Ratio (users/bins): 14.17


### Method 2: Demonstrating Collisions

In [6]:
print("\n" + "="*60)
print("Method 2: Understanding Hash Collisions")
print("="*60)

# Hash small set to show collisions
sample_users = data['user_id'].unique()[:20]
n_hash_bins = 8

print(f"\nHashing {len(sample_users)} users into {n_hash_bins} bins:")
print("\nUser ID           | Hash Value | Bin | Sign")
print("-" * 55)

bin_mapping = {}
for user in sample_users:
    bin_idx, sign = hash_feature(user, n_hash_bins, use_sign=True)
    hash_val = int(hashlib.md5(str(user).encode()).hexdigest(), 16)
    
    if bin_idx not in bin_mapping:
        bin_mapping[bin_idx] = []
    bin_mapping[bin_idx].append(user)
    
    sign_str = '+' if sign == 1 else '-'
    print(f"{user:15s}   | {hash_val:10d} | {bin_idx:3d} | {sign_str:4s}")

print("\nBins with multiple users (collisions):")
for bin_idx, users in sorted(bin_mapping.items()):
    if len(users) > 1:
        print(f"  Bin {bin_idx}: {users}")

print("\n💡 Collisions are inevitable with hashing!")
print("   Use more bins or signed hashing to mitigate impact")


Method 2: Understanding Hash Collisions

Hashing 20 users into 8 bins:

User ID           | Hash Value | Bin | Sign
-------------------------------------------------------
user_860          | 46890087979752084689423767843166761763 |   3 | +   
user_3772         | 314510935341441205354259018851629285273 |   1 | -   
user_3092         | 287817678175381765884347969379941614587 |   3 | -   
user_466          | 111771102325685326850788807003362933990 |   6 | +   
user_4426         | 121683311390653521647415280925270333126 |   6 | +   
user_3444         | 239898825032748638649485231930185721683 |   3 | +   
user_3171         | 117592770906359773925260463176607627321 |   1 | -   
user_2919         | 107209335665371574842093788270327599730 |   2 | +   
user_130          | 188440981106619782076789646910645272416 |   0 | +   
user_1685         | 89843043358346553795344324679350528380 |   4 | -   
user_769          | 65724280019161380001681788889449455508 |   4 | +   
user_2391         | 1639897

### Method 3: Using sklearn's FeatureHasher

In [16]:

print("\n" + "="*60)
print("Method 3: sklearn FeatureHasher")
print("="*60)

# Prepare data for FeatureHasher
def prepare_for_hasher(df, columns):
    """Convert DataFrame to dict format for FeatureHasher"""
    for _, row in df[columns].iterrows():
        yield {col: str(row[col]) for col in columns}

# Create hasher
n_features = 64
hasher = FeatureHasher(n_features=n_features, input_type='dict')

# Apply to training data
train_data, test_data = train_test_split(data, test_size=0.25, random_state=42)

train_dicts = list(prepare_for_hasher(train_data, ['user_id', 'url']))
test_dicts = list(prepare_for_hasher(test_data, ['user_id', 'url']))

X_train_hashed = hasher.transform(train_dicts)
X_test_hashed = hasher.transform(test_dicts)

# Add numerical feature
from scipy.sparse import hstack
X_train_full = hstack([X_train_hashed, train_data[['clicks']]])
X_test_full = hstack([X_test_hashed, test_data[['clicks']]])

y_train = train_data['converted']
y_test = test_data['converted']

print(f"\nFeatureHasher results:")
print(f"  Original categories: {data['user_id'].nunique() + data['url'].nunique()}")
print(f"  Hashed features: {n_features}")
print(f"  Dimensionality reduction: {data['user_id'].nunique() + data['url'].nunique()} → {n_features}")
print(f"  Combined sparse matrix shape: {X_train_full.shape}")


Method 3: sklearn FeatureHasher

FeatureHasher results:
  Original categories: 1687
  Hashed features: 64
  Dimensionality reduction: 1687 → 64
  Combined sparse matrix shape: (750, 65)


In [20]:
# Train model
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_full, y_train)

train_acc = rf.score(X_train_full, y_train)
test_acc = accuracy_score(y_test, rf.predict(X_test_full))

print(f"\nRandom Forest with Hashed Features:")
print(f"  Training Accuracy: {train_acc:.4f}")
print(f"  Testing Accuracy: {test_acc:.4f}")


Random Forest with Hashed Features:
  Training Accuracy: 1.0000
  Testing Accuracy: 0.4800


In [22]:
# Method 4: Comparing Bin Sizes
print("\n" + "="*60)
print("Method 4: Impact of Hash Bin Size")
print("="*60)

bin_sizes = [16, 32, 64, 128, 256]
results = []

for n_feat in bin_sizes:
    hasher_temp = FeatureHasher(n_features=n_feat, input_type='dict')
    
    X_tr = hasher_temp.transform(train_dicts).toarray()
    X_te = hasher_temp.transform(test_dicts).toarray()
    
    # Add clicks
    X_tr = np.column_stack([X_tr, train_data['clicks'].values])
    X_te = np.column_stack([X_te, test_data['clicks'].values])
    
    rf_temp = RandomForestClassifier(n_estimators=50, random_state=42, max_depth=10)
    rf_temp.fit(X_tr, y_train)
    
    test_acc_temp = accuracy_score(y_test, rf_temp.predict(X_te))
    
    # Estimate collision rate
    unique_cats = data['user_id'].nunique() + data['url'].nunique()
    collision_rate = max(0, 1 - n_feat / unique_cats) if unique_cats > n_feat else 0
    
    results.append({
        'n_features': n_feat,
        'test_accuracy': test_acc_temp,
        'ratio_cats_to_bins': unique_cats / n_feat,
        'est_collision_rate': f"{collision_rate*100:.1f}%"
    })

results_df = pd.DataFrame(results)
print("\nHash Bin Size vs Performance:")
print(results_df.to_string(index=False))

print("\n💡 Sweet spot: n_features ≈ 2 × number of unique categories")
print("   Too few bins → more collisions → information loss")
print("   Too many bins → sparse features → overfitting risk")




Method 4: Impact of Hash Bin Size

Hash Bin Size vs Performance:
 n_features  test_accuracy  ratio_cats_to_bins est_collision_rate
         16          0.476          105.437500              99.1%
         32          0.464           52.718750              98.1%
         64          0.420           26.359375              96.2%
        128          0.488           13.179688              92.4%
        256          0.488            6.589844              84.8%

💡 Sweet spot: n_features ≈ 2 × number of unique categories
   Too few bins → more collisions → information loss
   Too many bins → sparse features → overfitting risk
